# AquaFlow — leak & burst detection model

This notebook trains and evaluates the leak detector step by step. It uses the same code as
`train_leak_model.py` (in `leakdetect.py`), so both give the same results.

**Data honesty:** the households are **simulated** — household behaviour from SIMDEUM
(Blokker et al., 2010), passed through a sensor model **calibrated on the physical AquaFlow loop**,
with labelled leaks injected. Real-world validation uses data exported from the loop (step 9).

Steps: setup → data → look at the data → features → cross-validation → compare with the current
rules → ablation → breakdowns → final model → real data.

## 1. Setup

In **Google Colab** this cell clones the repository and installs the requirements.
Run locally from `edge-ai/leak_detection/`, it uses your checkout as it is.
If the repository is private, Colab will ask for credentials: use a GitHub personal access token,
or upload the repo as a ZIP instead.

In [ ]:
import os, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/Research-Project'):
        !git clone -q https://github.com/ShewonGun/Research-Project.git /content/Research-Project
    os.chdir('/content/Research-Project/edge-ai/leak_detection')
    !pip install -q -r requirements.txt

MODEL_DIR = os.getcwd()
sys.path.insert(0, MODEL_DIR)
print('Working in', MODEL_DIR)

## 2. Data

The simulated households live in `edge-ai/data/simdeum/out/` (about 120 MB, not in git).

* **Option A (recommended):** generate them once on your PC
  (`python edge-ai/data/simdeum/generate_simdeum_households.py`), zip the `out` folder, upload
  `out.zip` to Google Drive under `MyDrive/aquaflow/`, and run the cell below.
* **Option B:** generate them in Colab in a *separate* notebook — pySIMDEUM needs older pandas/numpy
  (`pip install pysimdeum==1.0.1 "pandas<2.2" "numpy<2"`, then *Runtime → Restart session*),
  which clashes with this notebook's libraries.

In [ ]:
DATA_DIR = os.path.abspath(os.path.join(MODEL_DIR, '..', 'data', 'simdeum', 'out'))

if IN_COLAB and not os.path.exists(os.path.join(DATA_DIR, 'households.csv')):
    from google.colab import drive
    drive.mount('/content/drive')
    !unzip -q -o /content/drive/MyDrive/aquaflow/out.zip -d {os.path.dirname(DATA_DIR)}

assert os.path.exists(os.path.join(DATA_DIR, 'households.csv')), 'Simulated households not found — see Option A/B above'
print('Data:', DATA_DIR)

## 3. Load the households and look at the data

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
import leakdetect as ld
import train_leak_model as tlm

data_dir = Path(DATA_DIR)
homes = sorted(p.name for p in data_dir.iterdir() if (p / '1min.csv').exists())
data = {h: ld.load_household(data_dir / h) for h in homes}          # h -> (per-minute frame, anomaly events)
info = pd.read_csv(data_dir / 'households.csv').set_index('household_id')
events = pd.concat([e.assign(household=h) for h, (_, e) in data.items()])
print(f'{len(homes)} households, {sum(len(d) for d, _ in data.values()):,} minutes')
print(events['type'].value_counts().to_string())
info[['house_type', 'routine', 'people', 'members', 'loop_litres_per_day']]

In [ ]:
# One day of one household: flow per minute, with any anomaly shaded.
h = 'H01'
df, ev = data[h]
day = df.index[0] + pd.Timedelta(days=2)
w = df.loc[day: day + pd.Timedelta(days=1)]
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(w.index.tz_convert(ld.LOCAL_TZ).tz_localize(None), w['flowRateMean'], color=ld.BLUE, lw=1)
for e in ev.itertuples():
    if e.end >= w.index[0] and e.start <= w.index[-1]:
        ax.axvspan(e.start.tz_convert(ld.LOCAL_TZ).tz_localize(None), e.end.tz_convert(ld.LOCAL_TZ).tz_localize(None),
                   color=ld.ORANGE, alpha=0.15, lw=0)
ld._style(ax, f'{h}: one day of flow (orange = anomaly)', 'L/min')
plt.show()

## 4. Features — computed from the past only

At minute *t* the model only sees minutes ≤ *t*, so it could run live. Four groups:

* **continuity** — how long water has flowed without stopping, how long since the house was last quiet,
  share of idle minutes, on/off switching. *The core leak signal: it doesn't depend on flow level or time.*
* **level** — how much water flows.
* **time** — hour of day.
* **baseline** — the same signals compared with **this house's own previous week** (the house-specific part).

In [ ]:
feats = {h: ld.make_features(data[h][0]) for h in homes}
for g, cols in ld.FEATURE_GROUPS.items():
    print(f'{g:<11}', ', '.join(cols))
feats['H01'].describe().T[['mean', 'min', 'max']].round(2).head(12)

## 5. Cross-validation by household

6 folds of 5 households. In each fold the model trains on 20 households, tunes its alarms on 5 others,
and is **tested on 5 households it has never seen**. Every household is tested once, and results are
reported as **mean ± standard deviation across folds**.

Two alarms are tuned separately on the validation households, each kept under a false-alarm budget:
a **burst alarm** (urgent) and a **micro-leak alarm**.

In [ ]:
FA_BUDGET = 0.05  # max false alarms per day, per alarm, when tuning
rules = {h: ld.rule_baseline(data_dir / h / 'readings.csv.gz', data[h][0].index) for h in homes}

experiments = {}
for name, kind in [('Random Forest', 'rf'), ('Gradient boosting', 'hgb'),
                   ('Isolation Forest (no labels)', 'iforest'), ('Current backend rules', 'rules')]:
    experiments[name] = tlm.cross_validate(kind, 'all', homes, feats, data, rules, FA_BUDGET, 0,
                                           keep_alarms=kind != 'rules')
    print(name, 'done')

cols_show = ['detector', 'micro_leak_recall', 'burst_recall', 'false_alarms_per_day',
             'micro_leak_delay_min', 'burst_delay_min', 'caught_total']
rows = [tlm.headline(n, cv) for n, cv in experiments.items()]
pd.DataFrame(rows)[cols_show]

## 6. Models vs the current rules

In [ ]:
comp = pd.DataFrame(rows).set_index('detector')
comp.index = ['Random\nForest', 'Gradient\nboosting', 'Isolation\nForest', 'Current\nrules']
os.makedirs('results/charts', exist_ok=True)
ld.chart_bars(comp, ['_micro', '_burst'], ['Micro-leaks caught', 'Bursts caught'],
              'Anomalies caught on unseen households (6-fold CV)', 'results/charts/1_detection_vs_rules.png',
              'Share of events caught', ylim=(0, 1.15))
ld.chart_bars(comp, ['_fa'], ['False alarms/day'], 'False alarms per day (lower is better)',
              'results/charts/2_false_alarms_by_detector.png', 'False alarms per day', fmt='{:.2f}')
display(Image('results/charts/1_detection_vs_rules.png')); display(Image('results/charts/2_false_alarms_by_detector.png'))

## 7. Ablation — what does the best model rely on?

If results hold **without time of day** and **without flow level**, the model detects leaks by
*continuity* — which is what transfers to households with unusual routines (e.g. night-shift workers).

In [ ]:
supervised = [r for r in rows if r['detector'] in ('Random Forest', 'Gradient boosting')]
best = max(supervised, key=lambda r: ((r['_micro'] + r['_burst']) / 2, -r['_fa']))
best_name = best['detector']; best_kind = 'rf' if best_name == 'Random Forest' else 'hgb'
print('Best supervised model:', best_name)

ablation = {'all features': experiments[best_name]}
for fs, label in [('no_time', 'without time of day'), ('no_time_no_level', 'without time & flow level'),
                  ('no_baseline', 'without house baseline')]:
    ablation[label] = tlm.cross_validate(best_kind, fs, homes, feats, data, rules, FA_BUDGET, 0)
abl_rows = [tlm.headline(n, cv) for n, cv in ablation.items()]
pd.DataFrame(abl_rows)[cols_show]

## 8. Breakdowns — per household (night-shift vs day) and per anomaly shape

In [ ]:
cv = experiments[best_name]
ph = []
for h, r in cv['per_house'].items():
    s = ld.summarise([r])
    ph.append({'household': h, 'house_type': info.loc[h, 'house_type'], 'routine': info.loc[h, 'routine'],
               'false_alarms_per_day': round(s['false_alarms_per_day'], 3),
               'micro_leaks_caught': s['micro_leak_caught'], 'bursts_caught': s['burst_caught']})
ph = pd.DataFrame(ph)
print('False alarms/day by routine:', ph.groupby('routine')['false_alarms_per_day'].mean().round(3).to_dict())
shapes = pd.DataFrame([e for r in cv['per_house'].values() for e in r['events']])
by_shape = shapes.groupby('type').agg(caught=('caught', 'sum'), total=('caught', 'size'),
                                      median_delay_min=('delay_min', 'median'))
by_shape['recall'] = (by_shape['caught'] / by_shape['total']).round(2)
display(by_shape)
ph

In [ ]:
# Example: a leaking cistern in a test household, with each alarm.
for h in homes:
    ev = data[h][1]; ev = ev[ev['type'] == 'cistern_leak']
    if len(ev):
        al = cv['alarms'][h]
        ld.chart_trace(data[h][0], ev.iloc[[0]], {'micro-leak alarm': al['micro_leak'], 'burst alarm': al['burst'],
                       'current rules': rules[h]}, f'{h}: leaking cistern — flow, true leak and alarms',
                       'results/charts/7_trace_cistern_leak.png')
        display(Image('results/charts/7_trace_cistern_leak.png'))
        break

## 9. Final model, saved — and tested on other data

Trained on H01–H25, alarms tuned on H26–H30. Then tested on:
* the **original single-household dataset** (made by a different simulator), and
* **real loop data**, if you export it: MongoDB → `readings` collection → filter your device → Export CSV
  (columns `timestamp, flowRate, volume`). Optionally list staged anomalies (e.g. a 20-minute pump run =
  burst) in a CSV with columns `start, end, type, label`.

In [ ]:
import joblib
model, cols, tuned = tlm.fit_and_tune(best_kind, 'no_baseline', feats, data, homes[:-5], homes[-5:], FA_BUDGET, 0)
joblib.dump({'model': model, 'features': cols, 'alarms': tuned, 'model_type': best_name}, 'results/leak_model.joblib')
print('Alarm settings (threshold, consecutive minutes):', tuned)

orig = Path(MODEL_DIR).parent / 'data' / 'loop_household' / 'out'
for mcsv in sorted(orig.glob('*_1min.csv')):
    df, ev = ld.load_single_file(mcsv, mcsv.with_name(mcsv.name.replace('_1min', '_events')))
    res = tlm.evaluate_house(best_kind, tlm.detect(best_kind, model, tuned, ld.make_features(df)[cols]), df.index, ev)
    print('Original single-household dataset:', {k: v for k, v in ld.summarise([res]).items() if 'caught' in k or 'false' in k})

REAL_READINGS = None          # e.g. '/content/drive/MyDrive/aquaflow/esp32_readings.csv'
REAL_EVENTS = None            # e.g. '/content/drive/MyDrive/aquaflow/staged_events.csv' (or None)
if REAL_READINGS:
    df, ev = ld.load_real_readings(REAL_READINGS, REAL_EVENTS)
    res = tlm.evaluate_house(best_kind, tlm.detect(best_kind, model, tuned, ld.make_features(df)[cols]), df.index, ev)
    print('REAL loop data:', ld.summarise([res]))

In [ ]:
# Save results to Google Drive (Colab deletes everything when the session ends).
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/aquaflow
    !cp -r results /content/drive/MyDrive/aquaflow/
    print('Saved to MyDrive/aquaflow/results')